# Mean Pooling

In [ ]:
!pip install -q sentence-transformers

import json
import os
import glob
import numpy as np
import torch
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, precision_score, recall_score, f1_score

BASE_PATH = "/kaggle/input/datasets/thaiphonghuan/multi-view-dataset/dataset_structuring"
TRAIN_PATH = os.path.join(BASE_PATH, "train")
VAL_PATH = os.path.join(BASE_PATH, "val")
TEST_PATH = os.path.join(BASE_PATH, "test")

CV_VIEWS = ['skills', 'education', 'year_experience', 'experience', 'responsibilities', 
            'projects', 'publications', 'achievement', 'certification', 'employment_history']
JD_VIEWS = ['skills', 'education', 'year_experience_requirement', 'experience_requirement', 'responsibilities']

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sbert_model = SentenceTransformer('BAAI/bge-base-en-v1.5').to(device)
#SentenceTransformer('all-mpnet-base-v2')
#SentenceTransformer('BAAI/bge-base-en-v1.5')
#SentenceTransformer('thenlper/gte-base')
#SentenceTransformer('jjzha/jobbert-base-cased')
#SentenceTransformer('swaraj20/resume_gte_embedding')
#SentenceTransformer('lwolfrum2/careerbert-g')

In [ ]:
def process_view(view_data):
    """Xử lý missing_section thành special token"""
    if view_data is None: return "[EMPTY_SECTION]", 0
    if isinstance(view_data, list):
        clean_list = [str(x) for x in view_data if str(x).lower() != "missing_section"]
        if not clean_list: return "[EMPTY_SECTION]", 0
        return ". ".join(clean_list), 1
    val = str(view_data).lower()
    if val == "missing_section" or val == "" or val == "none":
        return "[EMPTY_SECTION]", 0
    return str(view_data), 1

def run_mean_pooling_baseline(path):
    all_scores = []
    reg_labels = []
    bin_labels = []
    
    # Lấy danh sách file
    matched_files = glob.glob(os.path.join(path, "matched/*.json"))
    mismatch_files = glob.glob(os.path.join(path, "mismatch/*.json"))
    configs = [(matched_files, 1), (mismatch_files, 0)]
    
    print(f"Đang tính toán Mean Pooling Similarity cho: {path}")
    for files, b_label in configs:
        for file_path in tqdm(files, desc=f"Folder {b_label}"):
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # 1. Xử lý CV Views
            cv_data = [process_view(data['resume_structured'].get(v)) for v in CV_VIEWS]
            cv_texts, cv_masks = zip(*cv_data)
            
            # 2. Xử lý JD Views
            jd_data = [process_view(data['job_description_structured'].get(v)) for v in JD_VIEWS]
            jd_texts, jd_masks = zip(*jd_data)
            
            with torch.no_grad():
                # Encoding tất cả views
                cv_emb = sbert_model.encode(list(cv_texts), convert_to_tensor=True) # [10, d]
                jd_emb = sbert_model.encode(list(jd_texts), convert_to_tensor=True) # [5, d]
                
                # THỰC HIỆN MEAN POOLING (Chỉ tính trung bình các view có mask=1)
                # Tính cho CV
                cv_mask_tensor = torch.tensor(cv_masks, device=device).unsqueeze(1) # [10, 1]
                z_cv = (cv_emb * cv_mask_tensor).sum(dim=0) / (cv_mask_tensor.sum() + 1e-9)
                
                # Tính cho JD
                jd_mask_tensor = torch.tensor(jd_masks, device=device).unsqueeze(1) # [5, 1]
                z_jd = (jd_emb * jd_mask_tensor).sum(dim=0) / (jd_mask_tensor.sum() + 1e-9)
                
                # 3. TÍNH COSINE SIMILARITY CUỐI CÙNG giữa 2 vector đại diện
                score = util.cos_sim(z_cv, z_jd).item()
            
            all_scores.append(score)
            reg_labels.append(data['target_label'] / 10.0)
            bin_labels.append(b_label)
            
    return np.array(all_scores), np.array(reg_labels), np.array(bin_labels)

y_pred_test, y_true_reg, y_true_bin = run_mean_pooling_baseline(TEST_PATH)

In [ ]:
threshold = 0.5
y_pred_bin = (y_pred_test >= threshold).astype(int)

# Tính toán Metrics
mae = mean_absolute_error(y_true_reg, y_pred_test)
mse = mean_squared_error(y_true_reg, y_pred_test)
acc = accuracy_score(y_true_bin, y_pred_bin)
pre = precision_score(y_true_bin, y_pred_bin)
rec = recall_score(y_true_bin, y_pred_bin)
f1 = f1_score(y_true_bin, y_pred_bin)

print("\n" + "="*50)
print("KẾT QUẢ MEAN POOLING BASELINE (NO CROSS-VIEW)")
print("="*50)
print(f"Regression MAE:   {mae:.4f}")
print(f"Regression MSE:   {mse:.4f}")
print(f"Classification Acc: {acc*100:.2f}%")
print(f"Precision (Match): {pre:.4f}")
print(f"Recall (Match):    {rec:.4f}")
print(f"F1-Score:          {f1:.4f}")
print("="*50)

print("\nĐối chiếu mẫu:")
for i in range(5):
    print(f"Thực tế: {'Matched' if y_true_bin[i]==1 else 'Mismatch'} | Dự đoán Sim: {y_pred_test[i]:.4f}")

# Max Pooling

In [ ]:
!pip install -q sentence-transformers

import json
import os
import glob
import numpy as np
import torch
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, precision_score, recall_score, f1_score

BASE_PATH = "/kaggle/input/datasets/thaiphonghuan/multi-view-dataset/dataset_structuring"
TRAIN_PATH = os.path.join(BASE_PATH, "train")
VAL_PATH = os.path.join(BASE_PATH, "val")
TEST_PATH = os.path.join(BASE_PATH, "test")

CV_VIEWS = ['skills', 'education', 'year_experience', 'experience', 'responsibilities', 
            'projects', 'publications', 'achievement', 'certification', 'employment_history']
JD_VIEWS = ['skills', 'education', 'year_experience_requirement', 'experience_requirement', 'responsibilities']

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sbert_model = SentenceTransformer('BAAI/bge-base-en-v1.5').to(device)
#SentenceTransformer('all-mpnet-base-v2')
#SentenceTransformer('BAAI/bge-base-en-v1.5')
#SentenceTransformer('thenlper/gte-base')
#SentenceTransformer('jjzha/jobbert-base-cased')
#SentenceTransformer('swaraj20/resume_gte_embedding')
#SentenceTransformer('lwolfrum2/careerbert-g')

In [ ]:
def process_view(view_data):
    if view_data is None: return "[EMPTY_SECTION]", 0
    if isinstance(view_data, list):
        clean_list = [str(x) for x in view_data if str(x).lower() != "missing_section"]
        if not clean_list: return "[EMPTY_SECTION]", 0
        return ". ".join(clean_list), 1
    val = str(view_data).lower()
    if val == "missing_section" or val == "" or val == "none":
        return "[EMPTY_SECTION]", 0
    return str(view_data), 1

def run_max_pooling_baseline(path):
    all_scores = []
    reg_labels = []
    bin_labels = []
    
    matched_files = glob.glob(os.path.join(path, "matched/*.json"))
    mismatch_files = glob.glob(os.path.join(path, "mismatch/*.json"))
    configs = [(matched_files, 1), (mismatch_files, 0)]
    
    print(f"Đang tính toán Max Pooling Similarity cho: {path}")
    for files, b_label in configs:
        for file_path in tqdm(files, desc=f"Folder {b_label}"):
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            cv_data = [process_view(data['resume_structured'].get(v)) for v in CV_VIEWS]
            cv_texts, cv_masks = zip(*cv_data)
            
            jd_data = [process_view(data['job_description_structured'].get(v)) for v in JD_VIEWS]
            jd_texts, jd_masks = zip(*jd_data)
            
            with torch.no_grad():
                cv_emb = sbert_model.encode(list(cv_texts), convert_to_tensor=True) # [10, d]
                jd_emb = sbert_model.encode(list(jd_texts), convert_to_tensor=True) # [5, d]
                
                # CHUẨN BỊ MASK TENSOR
                cv_mask_tensor = torch.tensor(cv_masks, device=device).unsqueeze(1) # [10, 1]
                jd_mask_tensor = torch.tensor(jd_masks, device=device).unsqueeze(1) # [5, 1]

                # THỰC HIỆN MAX POOLING
                # Với những view có mask=0 (thiếu), ta gán giá trị cực thấp (-1e9) 
                z_cv = torch.max(cv_emb + (1 - cv_mask_tensor) * -1e9, dim=0)[0]
                z_jd = torch.max(jd_emb + (1 - jd_mask_tensor) * -1e9, dim=0)[0]
                
                # Tính Cosine Similarity giữa 2 vector 
                score = util.cos_sim(z_cv, z_jd).item()
            
            all_scores.append(score)
            reg_labels.append(data['target_label'] / 10.0)
            bin_labels.append(b_label)
            
    return np.array(all_scores), np.array(reg_labels), np.array(bin_labels)

y_pred_test, y_true_reg, y_true_bin = run_max_pooling_baseline(TEST_PATH)

In [ ]:
threshold = 0.5
y_pred_bin = (y_pred_test >= threshold).astype(int)

# Tính toán Metrics
mae = mean_absolute_error(y_true_reg, y_pred_test)
mse = mean_squared_error(y_true_reg, y_pred_test)
acc = accuracy_score(y_true_bin, y_pred_bin)
pre = precision_score(y_true_bin, y_pred_bin)
rec = recall_score(y_true_bin, y_pred_bin)
f1 = f1_score(y_true_bin, y_pred_bin)

print("\n" + "="*50)
print("KẾT QUẢ MAX POOLING BASELINE (NO CROSS-VIEW)")
print("="*50)
print(f"Regression MAE:   {mae:.4f}")
print(f"Regression MSE:   {mse:.4f}")
print(f"Classification Acc: {acc*100:.2f}%")
print(f"Precision (Match): {pre:.4f}")
print(f"Recall (Match):    {rec:.4f}")
print(f"F1-Score:          {f1:.4f}")
print("="*50)

print("\nĐối chiếu mẫu:")
for i in range(5):
    print(f"Thực tế: {'Matched' if y_true_bin[i]==1 else 'Mismatch'} | Dự đoán Sim: {y_pred_test[i]:.4f}")

# Attention Pooling

In [ ]:
!pip install -q sentence-transformers

import json
import os
import glob
import numpy as np
import torch
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, precision_score, recall_score, f1_score

BASE_PATH = "/kaggle/input/datasets/thaiphonghuan/multi-view-dataset/dataset_structuring"
TRAIN_PATH = os.path.join(BASE_PATH, "train")
VAL_PATH = os.path.join(BASE_PATH, "val")
TEST_PATH = os.path.join(BASE_PATH, "test")

CV_VIEWS = ['skills', 'education', 'year_experience', 'experience', 'responsibilities', 
            'projects', 'publications', 'achievement', 'certification', 'employment_history']
JD_VIEWS = ['skills', 'education', 'year_experience_requirement', 'experience_requirement', 'responsibilities']

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sbert_model = SentenceTransformer('BAAI/bge-base-en-v1.5').to(device)
#SentenceTransformer('all-mpnet-base-v2')
#SentenceTransformer('BAAI/bge-base-en-v1.5')
#SentenceTransformer('thenlper/gte-base')
#SentenceTransformer('jjzha/jobbert-base-cased')
#SentenceTransformer('swaraj20/resume_gte_embedding')
#SentenceTransformer('lwolfrum2/careerbert-g')

In [ ]:
def process_view(view_data):
    """Xử lý missing_section thành special token"""
    if view_data is None: return "[EMPTY_SECTION]", 0
    if isinstance(view_data, list):
        clean_list = [str(x) for x in view_data if str(x).lower() != "missing_section"]
        if not clean_list: return "[EMPTY_SECTION]", 0
        return ". ".join(clean_list), 1
    val = str(view_data).lower()
    if val == "missing_section" or val == "" or val == "none":
        return "[EMPTY_SECTION]", 0
    return str(view_data), 1

def prepare_pooling_data(path):
    cv_embs_list = []
    cv_masks_list = []
    jd_embs_list = []
    jd_masks_list = []
    reg_labels = []
    bin_labels = []
    
    matched_files = glob.glob(os.path.join(path, "matched/*.json"))
    mismatch_files = glob.glob(os.path.join(path, "mismatch/*.json"))
    configs = [(matched_files, 1), (mismatch_files, 0)]
    
    print(f"Đang tiền tính toán Embeddings cho: {path}")
    for files, b_label in configs:
        for file_path in tqdm(files, desc=f"Folder {b_label}"):
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            cv_data = [process_view(data['resume_structured'].get(v)) for v in CV_VIEWS]
            jd_data = [process_view(data['job_description_structured'].get(v)) for v in JD_VIEWS]
            
            cv_texts, cv_masks = zip(*cv_data)
            jd_texts, jd_masks = zip(*jd_data)
            
            with torch.no_grad():
                cv_emb = sbert_model.encode(list(cv_texts), convert_to_tensor=True).cpu().numpy()
                jd_emb = sbert_model.encode(list(jd_texts), convert_to_tensor=True).cpu().numpy()
            
            cv_embs_list.append(cv_emb)
            cv_masks_list.append(cv_masks)
            jd_embs_list.append(jd_emb)
            jd_masks_list.append(jd_masks)
            reg_labels.append(data['target_label'] / 10.0)
            bin_labels.append(b_label)
            
    return np.array(cv_embs_list), np.array(cv_masks_list), \
           np.array(jd_embs_list), np.array(jd_masks_list), \
           np.array(reg_labels), np.array(bin_labels)

cv_h_train, cv_m_train, jd_h_train, jd_m_train, y_train, b_train = prepare_pooling_data(TRAIN_PATH)
cv_h_val, cv_m_val, jd_h_val, jd_m_val, y_val, b_val = prepare_pooling_data(VAL_PATH)
cv_h_test, cv_m_test, jd_h_test, jd_m_test, y_test, b_test = prepare_pooling_data(TEST_PATH)

print("\nHoàn tất tiền tính toán Embeddings!")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader 

class PrecomputedPoolingDataset(Dataset):
    def __init__(self, cv_h, cv_m, jd_h, jd_m, y_reg, y_bin):
        self.cv_h = torch.tensor(cv_h, dtype=torch.float32)
        self.cv_m = torch.tensor(cv_m, dtype=torch.float32)
        self.jd_h = torch.tensor(jd_h, dtype=torch.float32)
        self.jd_m = torch.tensor(jd_m, dtype=torch.float32)
        self.y_reg = torch.tensor(y_reg, dtype=torch.float32).view(-1, 1)
        self.y_bin = torch.tensor(y_bin, dtype=torch.float32).view(-1, 1)

    def __len__(self): return len(self.y_reg)

    def __getitem__(self, idx):
        return self.cv_h[idx], self.cv_m[idx], self.jd_h[idx], self.jd_m[idx], \
               self.y_reg[idx], self.y_bin[idx]

class AttentionPoolingBaseline(nn.Module):
    def __init__(self, d_model=768):
        super().__init__()
        self.w = nn.Linear(d_model, 1, bias=False)

    def pool(self, h, mask):
        attn_scores = self.w(h).squeeze(-1)
        attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        alpha = F.softmax(attn_scores, dim=-1)
        z = torch.bmm(alpha.unsqueeze(1), h).squeeze(1)
        return z

    def forward(self, cv_h, cv_m, jd_h, jd_m):
        z_cv = self.pool(cv_h, cv_m)
        z_jd = self.pool(jd_h, jd_m)
        cos_sim = F.cosine_similarity(z_cv, z_jd)
        return cos_sim.unsqueeze(-1)

train_loader = DataLoader(PrecomputedPoolingDataset(cv_h_train, cv_m_train, jd_h_train, jd_m_train, y_train, b_train), batch_size=32, shuffle=True)
val_loader = DataLoader(PrecomputedPoolingDataset(cv_h_val, cv_m_val, jd_h_val, jd_m_val, y_val, b_val), batch_size=32)
test_loader = DataLoader(PrecomputedPoolingDataset(cv_h_test, cv_m_test, jd_h_test, jd_m_test, y_test, b_test), batch_size=32)

embedding_dim = cv_h_train.shape[-1] 
model = AttentionPoolingBaseline(d_model=embedding_dim).to(device)
criterion_reg = nn.MSELoss()
criterion_bin = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
epochs = 150
best_val_loss = float('inf')

print("Bắt đầu huấn luyện Attention Pooling (Pre-computed)...")

for epoch in range(epochs):
    model.train()
    total_train_loss = 0
    for cv_h, cv_m, jd_h, jd_m, y_reg, y_bin in train_loader:
        cv_h, cv_m, jd_h, jd_m = cv_h.to(device), cv_m.to(device), jd_h.to(device), jd_m.to(device)
        y_reg, y_bin = y_reg.to(device), y_bin.to(device)
        
        optimizer.zero_grad()
        pred_score = model(cv_h, cv_m, jd_h, jd_m)
        pred_score_clamped = torch.clamp(pred_score, 0, 1)
        
        loss = criterion_reg(pred_score_clamped, y_reg) + 0.5 * criterion_bin(pred_score_clamped, y_bin)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
    
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for cv_h, cv_m, jd_h, jd_m, y_reg, y_bin in val_loader:
            cv_h, cv_m, jd_h, jd_m = cv_h.to(device), cv_m.to(device), jd_h.to(device), jd_m.to(device)
            y_reg, y_bin = y_reg.to(device), y_bin.to(device)
            out = model(cv_h, cv_m, jd_h, jd_m)
            out_c = torch.clamp(out, 0, 1)
            val_loss += (criterion_reg(out_c, y_reg) + 0.5 * criterion_bin(out_c, y_bin)).item()
    
    avg_val = val_loss / len(val_loader)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d} | Train Loss: {total_train_loss/len(train_loader):.6f} | Val Loss: {avg_val:.6f}")
    
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), "attn_pooling_best_model.pth")

print("Hoàn tất!")

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, precision_score, recall_score, f1_score
import torch

model.load_state_dict(torch.load("attn_pooling_best_model.pth"))
model.eval()

y_preds_test = []
y_true_reg_test = []
y_true_bin_test = []

print("Đang thực hiện đánh giá trên tập Test (Attention Pooling)...")

with torch.no_grad():
    for cv_h, cv_m, jd_h, jd_m, y_reg, y_bin in test_loader:
        cv_h, cv_m = cv_h.to(device), cv_m.to(device)
        jd_h, jd_m = jd_h.to(device), jd_m.to(device)
        
        # Forward pass
        outputs = model(cv_h, cv_m, jd_h, jd_m)
        
        # Ép điểm về khoảng [0, 1] để tính toán chính xác
        outputs_clamped = torch.clamp(outputs, 0, 1)
        
        y_preds_test.extend(outputs_clamped.cpu().numpy().flatten())
        y_true_reg_test.extend(y_reg.numpy().flatten())
        y_true_bin_test.extend(y_bin.numpy().flatten())

y_preds_test = np.array(y_preds_test)
y_true_reg_test = np.array(y_true_reg_test)
y_true_bin_test = np.array(y_true_bin_test)

threshold = 0.5
y_preds_bin_test = (y_preds_test >= threshold).astype(int)

mae = mean_absolute_error(y_true_reg_test, y_preds_test)
mse = mean_squared_error(y_true_reg_test, y_preds_test)
acc = accuracy_score(y_true_bin_test, y_preds_bin_test)
pre = precision_score(y_true_bin_test, y_preds_bin_test)
rec = recall_score(y_true_bin_test, y_preds_bin_test)
f1 = f1_score(y_true_bin_test, y_preds_bin_test)

print("\n" + "="*60)
print("KẾT QUẢ ĐÁNH GIÁ: ATTENTION POOLING BASELINE")
print("="*60)
print(f"{'Regression MAE:':<25} {mae:.4f}")
print(f"{'Regression MSE:':<25} {mse:.4f}")
print(f"{'Classification Acc:':<25} {acc*100:.2f}%")
print(f"{'Precision (Match):':<25} {pre:.4f}")
print(f"{'Recall (Match):':<25} {rec:.4f}")
print(f"{'F1-Score:':<25} {f1:.4f}")
print("="*60)

# Single-view Matching

In [ ]:
!pip install -q sentence-transformers

import json
import os
import glob
import re
import numpy as np
import torch
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, precision_score, recall_score, f1_score

BASE_PATH = "/kaggle/working/split_dataset_synced"
TRAIN_PATH = os.path.join(BASE_PATH, "train")
VAL_PATH = os.path.join(BASE_PATH, "val")
TEST_PATH = os.path.join(BASE_PATH, "test")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = 'all-mpnet-base-v2'
#SentenceTransformer('all-mpnet-base-v2')
#SentenceTransformer('BAAI/bge-base-en-v1.5')
#SentenceTransformer('thenlper/gte-base')
#SentenceTransformer('jjzha/jobbert-base-cased')
#SentenceTransformer('swaraj20/resume_gte_embedding')
#SentenceTransformer('lwolfrum2/careerbert-g')

print(f" Loading Model: {MODEL_NAME}...")
sbert_model = SentenceTransformer(MODEL_NAME).to(device)

def clean_text(text):
    """Tiền xử lý văn bản: Xóa dấu *, /, \n và các ký tự đặc biệt rác"""
    if not text: return ""
    # 1. Thay thế dấu xuống dòng và dấu xẹt bằng khoảng trắng
    text = re.sub(r'[\n\r/]', ' ', text)
    # 2. Xóa các dấu sao (thường dùng làm bullet points)
    text = text.replace('*', '')
    # 3. Xóa các khoảng trắng dư thừa
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

In [ ]:
def run_single_view_matching(path):
    all_scores = []
    reg_labels = []
    bin_labels = []
    
    matched_files = glob.glob(os.path.join(path, "matched/*.json"))
    mismatch_files = glob.glob(os.path.join(path, "mismatch/*.json"))
    configs = [(matched_files, 1), (mismatch_files, 0)]
    
    print(f" Đang tính toán Single-view Similarity cho: {path}")
    for files, b_label in configs:
        for file_path in tqdm(files, desc=f"Folder {actual_folder(b_label)}"):
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # 1. Lấy và Gộp dữ liệu JD (Description + Additional Info)
            jd_input = data.get('input', {})
            jd_raw = jd_input.get('job_description', "") + " " + jd_input.get('additional_info', "")
            jd_cleaned = clean_text(jd_raw)
            
            # 2. Lấy dữ liệu CV (Resume)
            cv_raw = jd_input.get('resume', "")
            cv_cleaned = clean_text(cv_raw)
            
            # 3. Lấy nhãn (Target Label từ aggregated macro score)
            # Lưu ý: Điểm này nằm trong output -> scores -> aggregated_scores
            gt_score = data.get('output', {}).get('scores', {}).get('aggregated_scores', {}).get('macro_scores', 0)
            
            with torch.no_grad():
                # Encoding duy nhất 1 vector cho mỗi bên
                cv_emb = sbert_model.encode(cv_cleaned, convert_to_tensor=True)
                jd_emb = sbert_model.encode(jd_cleaned, convert_to_tensor=True)
                
                # Cosine Similarity 
                score = util.cos_sim(cv_emb, jd_emb).item()
            
            all_scores.append(score)
            reg_labels.append(gt_score / 10.0) # Chuẩn hóa nhãn về [0, 1]
            bin_labels.append(b_label)
            
    return np.array(all_scores), np.array(reg_labels), np.array(bin_labels)

def actual_folder(label):
    return "Matched" if label == 1 else "Mismatch"

y_pred_test, y_true_reg, y_true_bin = run_single_view_matching(TEST_PATH)

In [ ]:
threshold = 0.5
y_pred_bin = (y_pred_test >= threshold).astype(int)

# Tính toán các Metrics
mae = mean_absolute_error(y_true_reg, y_pred_test)
mse = mean_squared_error(y_true_reg, y_pred_test)
acc = accuracy_score(y_true_bin, y_pred_bin)
pre = precision_score(y_true_bin, y_pred_bin)
rec = recall_score(y_true_bin, y_pred_bin)
f1 = f1_score(y_true_bin, y_pred_bin)

print("\n" + "="*60)
print("KẾT QUẢ: SINGLE-VIEW MATCHING BASELINE (HOLISTIC)")
print("="*60)
print(f"{'Regression MAE:':<25} {mae:.4f}")
print(f"{'Regression MSE:':<25} {mse:.4f}")
print(f"{'Classification Acc:':<25} {acc*100:.2f}%")
print(f"{'Precision (Match):':<25} {pre:.4f}")
print(f"{'Recall (Match):':<25} {rec:.4f}")
print(f"{'F1-Score:':<25} {f1:.4f}")
print("="*60)

print("\n CHI TIẾT 10 MẪU ĐẦU TIÊN:")
print(f"{'ID':<4} | {'Thực tế':<10} | {'Điểm Gốc':<10} | {'Dự đoán Sim':<12} | {'Trạng thái'}")
print("-" * 65)
for i in range(min(10, len(y_pred_test))):
    fld = "Matched" if y_true_bin[i] == 1 else "Mismatch"
    gt = y_true_reg[i]
    ps = y_pred_test[i]
    st = " ĐÚNG" if y_pred_bin[i] == y_true_bin[i] else " SAI"
    print(f"{i:<4} | {fld:<10} | {gt:<10.4f} | {ps:<12.4f} | {st}")